## 微调流程：
1. 解决什么问题？
- 训练模型/微调模型
- base model/chat model
2. 数据准备
   1. 中文/英文
   2. hf公开数据
   3. github
   4. paper
   5. 模型生成：
   
比如给定 100 个已有的 seed 数据，让 GPT4 等优质模型帮我们接着生成其他的类似的数据
（像 self-instruct）给定一些文章，把文章内容转换成<input, output>形式。 比如一个医学杂志，基于里面的文本，让 GPT4 帮我们说生成一问一答的形式，或者多轮对话形式。 举个简单的例子，给定一个文本：“这个感冒药一瓶的价格是 10 块钱”，可以转换成<input: 这瓶感冒药多少钱？,output: 10 块钱> ， 这样就构成了 instruction data格式。同时，也可以扩充成多轮对话，比如让 GPT4 接着帮我们生成问题 然而，GPT4 的价格一般是很贵的，一种这方案是使用Mistral 等高质量开源大模型来生成，这时候你的花销也就是推理成本。

- Base:
  - 如果我们有大量的专业数据要训练，那这个时候选择 Base 应该是比较合理的，因为我们希望
在没有被“污染”过的模型基础上接着训练。 
  - Base 模型是最干净的。 如果需要对话能力，我们也完全可以引入我们自己的对话数据来驱动大模型的对话能力。（比如做医疗的，那就收集大量医疗相关的对话数据）。
  - 不需要对话能力：如果只是追求语言理解能力，我们就可以选择 Base 来接着 finetune.
  - 深度定制：比如一个客户要解决金融领域的智能投顾问题，那这个时候从 Base 开始训练也是比较合理的选择。


- chat：
  - 对于需要理解和生成自然语言对话的应用（如聊天机器人、客户服务自动化、虚拟助手等），Chat 模型由于其在对话上的专⻔训练，可能会提供更好的性能。
  - 如果你的应用场景需要模型理解和维持⻓时间的对话上下文，Chat 模型通常更适合这种需求。
  - 资源有限，可以用 Chat 模型也是比较合理的选择，毕竟想驱动一个大模型的对话能力也是需要对话能力的训练的。

# 整体流程：
1. base模型下载
2. 数据准备/样本格式化
3. 配置模型:lora/tokenizer
4. 

# 模型下载

In [1]:
# 模型下载
# !pip install modelscope
import torch
from modelscope import snapshot_download, AutoModel, AutoTokenizer
import os
 
model_dir = snapshot_download('LLM-Research/Meta-Llama-3-8B-Instruct', 
                              cache_dir='cache', revision='master')

c:\Users\wxb55\.conda\envs\urban_climate\llm1\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# 导入模块

In [2]:
# 导入模块
# !pip install datasets peft bitsandbytes transformers trl pynvml
# peft: 用于微调（Lora等）
# bitsandbytes：专注于提供高效的位操作和字节级处理功能，尤其适用于深度学习和自然语言处理任务中对性能要求较高的场景。
#   这个库提供了一些底层操作，可以加速数据处理，尤其是在需要进行位打包、解包或进行位级逻辑运算时。
# trl: 主要用于SFT, reward model, 强化学习

import os
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, # 用于自动加载或下载因果语言模型（如 GPT 系列），这类模型适用于生成任务。
    AutoTokenizer, # 用于自动加载或下载与特定模型相对应的 tokenizer
    BitsAndBytesConfig, # 这通常是与 bitsandbytes 库相关的配置，用于设置模型训练时的低精度优化参数。
    TrainingArguments, # 包含了训练模型时所需的各种参数
    pipeline,
    logging,
)
from peft import LoraConfig
from trl import SFTTrainer
from tqdm import tqdm
import torch
import time
import pandas as pd
import numpy as np

# 数据导入

url:https://huggingface.co/datasets/neil-code/dialogsum-test

多轮对话总结任务

读取数据集后我们需要对数据进行处理，构建一个模型prompt：
  Give the conversation, extract the main points and summarize the conversions
{dialogue}
{summary}


In [3]:
dataset_name = "neil-code/dialogsum-test"
dataset = load_dataset(dataset_name)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})

In [4]:
def create_prompt_formats(sample):
    """
    格式化样本 ('instruction','output')
    :param sample: input data
    """

    # 构建 prompt
    INTRO_BLURB = "Instruct: Below is an instruction that describes a task. " \
    "Write a response that appropriately completes the request."
    INSTRUCTION_KEY = "Input: Please Summarize the below conversation."
    RESPONSE_KEY = "Output:"
    
    blurb = f"\n{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY}"
    input_context = f"{sample['dialogue']}" if sample["dialogue"] else None
    response = f"{RESPONSE_KEY}\n{sample['summary']}"
    
    parts = [part for part in [blurb, instruction, input_context, response] if part]
    formatted_prompt = "\n\n".join(parts)
    # prompt记录
    sample["text"] = formatted_prompt
    return sample

# 结果文件
print(create_prompt_formats(dataset['train'][0])['text']) # 后续只需要保留text部分，其他部分就不需要了



Instruct: Below is an instruction that describes a task. Write a response that appropriately completes the request.

Input: Please Summarize the below conversation.

#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medicatio

# 模型配置
在加载模型时使用QLORA量化为4bit，

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# =========================================================
# 1. 设置计算精度 (Compute DType)
# =========================================================
# 这里获取 torch.float16。
# 作用：虽然模型存储时会被压缩成 4-bit，但在 GPU 核心进行矩阵乘法运算的那一瞬间，
# 数据会被“解压”回 float16 进行计算，以保证数值稳定性。
compute_dtype = getattr(torch, "float16") 

# =========================================================
# 2. 定义量化配置 (BitsAndBytesConfig)
# =========================================================
quant_config = BitsAndBytesConfig(
    # [核心开关] 开启 4-bit 加载
    # 作用：将模型权重从 FP16 (16位) 压缩到 INT4 (4位)，显存占用直接砍到 1/4。
    load_in_4bit=True,  

    # [关键参数修正] 4-bit 量化类型
    # 原注释修正：nf4 不是"非功能性"，而是 "NormalFloat 4-bit" (正态浮点 4位)。
    # 原理：LLM 的权重通常服从正态分布。NF4 是一种基于分位数的量化数据类型，
    # 它在正态分布的数据上比标准的 FP4 或 INT4 精度更高，是 QLoRA 论文的核心创新之一。
    bnb_4bit_quant_type="nf4",  

    # [计算精度]
    # 作用：指定“解压”后运算时使用的精度。必须与后续 LoRA 层的精度一致 (通常为 float16)。
    bnb_4bit_compute_dtype=compute_dtype,

    # [极致省显存] 双重量化 (Double Quantization)
    # 原理：量化本身需要存储一些“常数”来记录缩放比例。
    # 这个参数是对这些“量化常数”再进行一次量化。
    # 效果：每个参数额外节省 0.4 bits 显存 (对于 65B 模型能省 3GB)，几乎无精度损失。
    bnb_4bit_use_double_quant=True, 
)

# =========================================================
# 3. 加载基座模型 (Base Model)
# =========================================================
model_path = "cache/LLM-Research/Meta-Llama-3-8B-Instruct/"

original_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    
    # 显式指定非量化层(如 LayerNorm)的精度
    torch_dtype=compute_dtype,
    
    # [设备映射]
    # {"": 0} 的含义：强制将模型的**所有层**都加载到 **GPU 0** 上。
    # 注意：在 QLoRA 训练中，这通常是为了防止 bitsandbytes 自动将模型切分到 CPU 或其他卡，
    # 导致与 PEFT 库配合时出现底层冲突。它主要用于单卡训练或手动控制的多卡环境。
    device_map={"": 0}, 
    
    # 传入上面的量化配置
    quantization_config=quant_config
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## 加载tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_path, 
                                          use_fast=False,  # 是否使用快速分词器
                                          trust_remote_code=True, 
                                          padding_side="left",
                                          add_eos_token=True, # 结束符， end of sentence
                                          add_bos_token=True  # 开始符， beginning of sentence
                                          # -> <bos>I love this cat<eos> 
                                          )
'''padding_side=left 的意思是，这个参数指定了在序列⻓度不足时，在序列的哪一侧添加填充
（padding）。在这里设置为"left"意味着填充将被添加到序列的左侧。比如我们在进行 mini-batch
训练的时候，为了达到最好的训练效率，会把 mini-batch 里面的 input⻓度弄成一样的。 一般的
操作是在 mini-batch 中，假如有 10 个不同的 input，而且每个 input ⻓度不一样，这时候可以选
择最⻓的作为标准，对于剩下的 input，⻓度不足的部分用 padding token 来填充，可以在序列的
右边添加，也可以在左边添加。
一般会选择left-padding （八股）
'''

# 将分词器的填充令牌（pad_token）设置为与结束符令牌（eos_token）相同。
tokenizer.pad_token_id = tokenizer.eos_token_id

# 对未经微调的模型进行测试
1.  构造模型的输入：这部分就是通过 template 来构造即可，结果发送给模型
2.  根据输入调用模型，并生成输出，最后再通过 tokenizer 把输出转化为字符串的形式（用户能看得
懂的形式)

In [9]:
# 构建一个测试用tokenizer
eval_tokenizer = AutoTokenizer.from_pretrained(model_path, add_bos_token=True, 
                                               trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

# 模型推理
def gen(model,p, maxlen=100, sample=True):
    '''
    将输入转换为token
    p: prompt
    '''
    toks = eval_tokenizer(p, return_tensors="pt") # 编码以pytorch的tensor返回
    # 使用generate生成结果
    res = model.generate(**toks.to("cuda"), 
                         max_new_tokens=maxlen,  # 最大token
                         do_sample=sample, # 是否在生成时采样，如果为 True，则在生成时使用概率分布，这使得生成的文本更多样化
                         num_return_sequences=1, # 生成序列数量
                         temperature=0.1, 
                         num_beams=1,# beam searchg， 为1就是贪心
                         top_p=0.95,
).to('cpu')
    return eval_tokenizer.batch_decode(res,skip_special_tokens=True)
    
# %%time
from transformers import set_seed
seed = 42
set_seed(seed)
index = 10
 
prompt = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']
formatted_prompt = f"Instruct: Summarize the following conversation.\nInput:{prompt}\nOutput:\n"
res = gen(original_model,formatted_prompt,100)
#print(res[0])
output = res[0].split('Output:\n')[1]
dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{formatted_prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Instruct: Summarize the following conversation.
Input:#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
Output:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# attends Brian's birth

# 模型微调
## 1. 数据预处理
- 对于给定的数据，进行格式化，使得满足 instruction 数据所需要的格式。
- 把文本转换成 token ids 的形式

In [ ]:
def get_max_length(model):
    # 找到模型关于最大长度的配置
    conf = model.config
    max_length = None
    for length_setting in ["n_positions", "max_position_embeddings", "seq_length"]:
        max_length = getattr(model.config, length_setting, None)
        if max_length:
            print(f"Found max lenth: {max_length}")
            break
        if not max_length:
            max_length = 1024
        print(f"Using default max length: {max_length}")
    return max_length
 
def preprocess_batch(batch, tokenizer, max_length):
    """
    Tokenizing a batch
    """
    return tokenizer(batch["text"],
                    max_length=max_length,
                    truncation=True, # 超过最大长度则截断
                    )

from functools import partial
def preprocess_dataset(tokenizer: AutoTokenizer, max_length: int,seed, dataset): 
    # Add prompt to each sample
    print("Preprocessing dataset...")
    dataset = dataset.map(create_prompt_formats) #  格式化映射
    _preprocessing_function = partial(preprocess_batch, max_length=max_length, tokenizer=tokenizer) # partial固定参数，映射到每一条目
    dataset = dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=['id', 'topic', 'dialogue', 'summary'],
    )
    # 过滤样本
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < 
max_length)
    # Shuffle 数据
    dataset = dataset.shuffle(seed=seed)
    print("Preprocessing dataset done.")
    return dataset


In [11]:
max_length = get_max_length(original_model)
print(max_length)
train_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset['train'])
eval_dataset = preprocess_dataset(tokenizer, max_length, seed, dataset['validation'])

Using default max length: 1024
Found max lenth: 8192
8192
Preprocessing dataset...


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1999 [00:00<?, ? examples/s]

Preprocessing dataset done.
Preprocessing dataset...


Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Filter:   0%|          | 0/499 [00:00<?, ? examples/s]

Preprocessing dataset done.


## 2.预训练模型配置
刚刚 load 进来的模型是预训练模型，是已经在大量的数据上训练的。 接下来，我们要在它的基础上做 finetune，这就意味着我们不会改变预训练模型本身的参数，而是训练一个新的模块来为下游任务进行调整。 原来模型的参数为 W，则新的模型参数为Δ W，那为了训练ΔW， 我们需要一个新的模型，包含相应的参数，同时我们要把原来的参数设置为不可训练。 通过 get_peft_model 就可以构造此模型。

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# =========================================================
# 1. 配置 LoRA (Low-Rank Adaptation)
# =========================================================
Loraconfig = LoraConfig(
    # --- 核心数学参数 ---
    # r (Rank, 秩): 决定了 LoRA 矩阵的“容量”或“厚度”。
    # 原理：LoRA 将大矩阵分解为两个小矩阵 A(d*r) 和 B(r*d)。
    # 数值：r 越大，可训练参数越多，模型拟合能力越强，但显存开销越大。
    # 经验：对于 7B 模型，16 或 32 是常见的平衡点；64 或 128 用于复杂任务。
    r=32, 

    # lora_alpha: 缩放系数 (Scaling Factor)。
    # 作用：控制 LoRA 权重对原模型权重的影响程度。
    # 公式：最终权重更新 = \Delta W \times (\alpha / r)。
    # 算账：这里 alpha=16, r=32，缩放比例 = 16/32 = 0.5。意味着 LoRA 的更新被打了个五折。
    # 技巧：通常设为 r 的一半或等于 r。
    lora_alpha=16, 

    # --- 作用范围 ---
    # target_modules: 指定在哪些层插入 LoRA 适配器。
    # 含义：这些是 Transformer 中 Self-Attention 层的四个投影矩阵。
    # 建议：如果显存允许，加上 MLP 层 (gate_proj, up_proj, down_proj) 通常效果更好。
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 

    # --- 训练细节 ---
    # bias: 是否训练偏置项。
    # "none": 全都不训练（最省显存）。
    # "all": 全部训练。
    # "lora_only": 只训练 LoRA 层的 bias。
    bias="none", 

    # lora_dropout: 随机失活率。
    # 作用：训练时随机让 1% 的神经元不工作，防止模型死记硬背（过拟合）。
    lora_dropout=0.01, 

    # task_type: 任务类型。
    # CAUSAL_LM: 因果语言建模（Causal Language Modeling），即 GPT 类的“预测下一个词”任务。
    task_type="CAUSAL_LM", 
)

# =========================================================
# 2. 模型预处理 (显存优化与稳定性)
# =========================================================

# [步骤 A] 开启梯度检查点 (Gradient Checkpointing)
# 解释：用“时间换空间”。
# 默认：保存每一层的激活值以计算梯度（显存占用大）。
# 开启后：不保存中间激活值，反向传播需要时重新算一遍（显存节省 50-70%，速度慢 20-30%）。
original_model.gradient_checkpointing_enable()

# [步骤 B] 为量化训练做准备 (Magic Function)
# 这是一个非常关键的工具函数，它默默做了三件事：
# 1. 冻结原模型的所有参数 (requires_grad=False)。
# 2. 将所有 LayerNorm 层强制转为 float32 数据类型（因为 int8/fp16 的 LayerNorm 容易溢出导致训练崩溃）。
# 3. 开启输入层的梯度需求 (require_grads)，防止报错。
original_model = prepare_model_for_kbit_training(original_model)

# =========================================================
# 3. 组装模型
# =========================================================
# 动作：在冻结的 original_model 上，挂载可训练的 LoRA 旁路。
# 结果：peft_model 现在包含了“巨大的冻结基座” + “小巧的可训练 LoRA”。
peft_model = get_peft_model(original_model, Loraconfig)

## 3. finetune模型

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer

output_dir = './cache/peft-dialogue-summary-training/final-checkpoint'

# =========================================================
# 1. 定义训练参数 (TrainingArguments)
# =========================================================
peft_training_args = TrainingArguments(
    # --- 文件与输出 ---
    output_dir = output_dir,        # 模型检查点(checkpoint)和最终模型的保存路径
    overwrite_output_dir = True,    # 强制覆盖：如果目录下有旧文件，直接覆盖，防止报错
    report_to = "none",             # 汇报工具：设为 "none" 表示不上传 WandB/TensorBoard，仅在本地记录
    
    # --- 训练进度控制 (关键：按步数而非轮数) ---
    # 区别：Epoch是把所有数据看一遍，Step是更新一次参数。
    # 场景：当数据量巨大（如几十万条）时，跑完1个Epoch太慢，通常设定 max_steps 来快速收敛。
    max_steps = 2000,               # 强制训练 2000 步后停止（无论是否看完数据集）
    warmup_steps = 1,               # 热身步数：前1步学习率从0线性升到 2e-4，防止起步梯度爆炸
    
    # --- 核心超参数 ---
    learning_rate = 2e-4,           # 学习率：LoRA 微调通常使用较大的 LR (2e-4)，全量微调通常用 1e-5
    
    # --- 显存优化三剑客 (Batch Size, Accumulation, Optim) ---
    # 策略：为了在单卡(如 16G/24G 显存)上跑起来，我们把单次输入压到最小，通过累积来模拟大 Batch。
    per_device_train_batch_size = 1,# 显存不够，设为1 (每次只读1条数据进显存)
    gradient_accumulation_steps = 1,# 梯度累积：如果是 4，则积累 4 次 loss 才更新一次参数。
                                    # 实际 Batch Size = 1 * 1 = 1 (非常小，建议显存允许的话设为 4 或 8)
    
    # [显存救星 1] 8-bit 分页优化器
    # 作用：将优化器状态量化为 8bit，并在显存不足时自动换入到 CPU 内存(Paged)。
    optim = "paged_adamw_8bit",     
    
    # [显存救星 2] 梯度检查点
    # 作用：不保存中间激活值，反向传播时重新计算。
    # 代价：训练速度慢 20%-30%，但显存节省 50%-70%。
    gradient_checkpointing = True,  
    
    # --- 日志与监控 ---
    logging_dir = "./logs",         # TensorBoard 日志存放位置
    logging_steps = 100,            # 每 100 步打印一次 Loss (比如 Step 100, 200...)
    
    # --- 保存与评估策略 ---
    save_strategy = "steps",        # 按步数保存（和 max_steps 对应）
    save_steps = 100,               # 每 100 步存一个 checkpoint
    do_eval = True,                 # 开启验证
    eval_steps = 100,               # 每 100 步在验证集上跑一次测试，看模型有没有变聪明
    
    # --- 效率优化 ---
    # 作用：将长度相近的句子放在同一个 Batch。
    # 原理：减少 Padding 的数量（短句不需要补很多 0），极大提升计算效率。
    group_by_length = True,         
    
    # --- [修复建议] 消除 PeftModel 警告 ---
    # 显式告诉 Trainer 数据集里哪个字段是标签，避免 PeftModel 隐藏参数导致 Trainer 找不到 Label
    label_names = ["labels"],       
)

# =========================================================
# 2. 模型配置调整 (防止冲突)
# =========================================================
# 为什么要设为 False？
# use_cache=True 是为了在推理(生成)时缓存 KV 值以加速。
# 但在训练时，它与 gradient_checkpointing 不兼容，必须关闭，否则会报错或显存溢出。
peft_model.config.use_cache = False

# 必须显式开启输入的梯度需求，否则 gradient checkpointing 会导致报错 "element 0 of tensors does not require grad"
peft_model.enable_input_require_grads()

# =========================================================
# 3. 初始化训练器 (Trainer)
# =========================================================
peft_trainer = transformers.Trainer(
    model=peft_model,               # 你的 PeftModel (基座模型 + LoRA 权重)
    train_dataset=train_dataset,    # 预处理好的训练集
    eval_dataset=eval_dataset,      # 预处理好的验证集
    args=peft_training_args,        # 上面定义的参数配置
    
    # --- 数据整理器 (Data Collator) ---
    # 核心作用：动态 Padding (Dynamic Padding)
    # 1. 它会看这一个 Batch 里最长的句子是多长 (比如 50)，然后把其他句子 Pad 到 50，而不是 1024。
    # 2. mlm=False：告诉它这是生成任务，它会自动把 input_ids 复制一份作为 labels。
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [16]:
peft_trainer.train()

Step,Training Loss
100,1.457300
200,1.356300
300,1.356300
400,1.343900
500,1.355500
600,1.274600
700,1.292900
800,1.295400
900,1.353900
1000,1.352500


TrainOutput(global_step=2000, training_loss=1.3341190338134765, metrics={'train_runtime': 1573.3515, 'train_samples_per_second': 1.271, 'train_steps_per_second': 1.271, 'total_flos': 2.366768522067149e+16, 'train_loss': 1.3341190338134765, 'epoch': 1.0005002501250626})

In [17]:
# 训练参数量计算
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\n \
                all model parameters: {all_model_params}\n \
                percentage of trainable model parameters:  \
                {100 * trainable_model_params / all_model_params:.2f}%"

In [ ]:
# Free memory 
del original_model
del peft_trainer
torch.cuda.empty_cache()

## 4. 导入lora微调模型
主要是和并lora训练参数和base模型

In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# =========================================================
# 1. 定义计算精度
# =========================================================
# 虽然模型平时是以 4-bit 存储的（为了省显存），但在进行矩阵乘法运算时，
# 显卡不支持 4-bit 直接计算，所以需要瞬间解压成 float16 来计算。
# getattr(torch, "float16") 等同于 torch.float16
compute_dtype = getattr(torch, "float16")

# =========================================================
# 2. 配置 4-bit 量化参数 (QLoRA 的精髓)
# =========================================================
quant_config = BitsAndBytesConfig(
    # 开启 4-bit 加载：这会将模型权重压缩到 4-bit。
    # 效果：模型体积缩小到原来的 1/4 左右（例如 8B 模型从 16GB 降到约 5GB）。
    load_in_4bit=True,
    
    # 量化类型 "nf4" (NormalFloat 4-bit)：
    # 这是 QLoRA 论文提出的一种专门针对正态分布权重优化的数据类型。
    # 相比普通的 "fp4"，它的精度损失更小，模型效果更好。
    bnb_4bit_quant_type="nf4",
    
    # 计算数据类型：
    # 关键点：存储用 4-bit，计算用 float16。
    # 流程：4-bit 权重 -> 解压为 float16 -> 运算 -> 结果。
    bnb_4bit_compute_dtype=compute_dtype,
    
    # 开启双重量化 (Double Quantization)：
    # 原理：对“量化常数”本身再进行一次量化。
    # 效果：每 1B 参数能额外节省约 0.4GB 显存，且几乎不影响精度。纯粹的“免费午餐”。
    bnb_4bit_use_double_quant=True,
)

model_path = "./cache/LLM-Research/Meta-Llama-3-8B-Instruct/"

# =========================================================
# 3. 加载量化模型 (Base Model)
# =========================================================
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    
    # 显式指定 torch_dtype，确保非量化层（如 LayerNorm）使用 float16 以保持稳定
    torch_dtype=compute_dtype,
    
    # 设备映射：
    # {"": 0} 意思是把模型的所有层都强制塞到 GPU 0 上。
    # 注意：在多卡环境如果不指定 map，bitsandbytes 可能会尝试切分模型，导致某些奇怪的 bug。
    device_map={"": 0},
    
    # 传入上面定义的量化配置
    quantization_config=quant_config
)

# =========================================================
# 4. 加载分词器 (Tokenizer)
# =========================================================
eval_tokenizer = AutoTokenizer.from_pretrained(
    model_path, 
    add_bos_token=True,    # 对 Llama 3 来说，加上 <bos> 通常有助于模型识别生成的起点
    trust_remote_code=True,# 允许执行模型仓库里的自定义代码（某些新模型需要）
    use_fast=False         # 使用 Python 实现的慢速分词器。
                           # 虽慢，但有时比 Rust 实现的 FastTokenizer 对特殊字符的处理更准确/兼容性更好。
)

# 设置 Padding Token：
# 因为 Llama 3 默认没有 pad token，我们借用 eos token 来充当 padding。
# 这就是我们之前讨论的：用“结束符”来代表“空白/无意义”。
eval_tokenizer.pad_token = eval_tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel

# =========================================================
# 加载微调后的模型 (Assembly: Base + LoRA)
# =========================================================
# 这一步就像给手机（Base Model）装上了一个新的 APP（LoRA），
# 让它具备了新的功能，但手机本身的硬件（权重）没有变。
ft_model = PeftModel.from_pretrained(
    model=base_model,  # 1. 你的基座模型 (Base Model)
                       # 这是之前加载的 Llama-3-8B（可能是 4-bit 量化的）。
                       # 它提供了语言模型的通用能力（语法、常识等）。
                       
    model_id="./cache/peft-dialogue-summary-training/final-checkpoint/checkpoint-500",
                       # 2. 你的 LoRA 权重路径 (Adapter Weights)
                       # 注意：这里加载的**不是**几 GB 的完整模型，而是你刚刚训练出来的“补丁包”。
                       # 文件里通常只有 adapter_model.bin 或 safetensors，体积很小。
                       # 'checkpoint-500' 表示你觉得第 500 步的模型效果最好，所以选了它。

    torch_dtype=torch.float16,
                       # 3. 适配器的精度
                       # 即使你的 Base Model 是 4-bit (nf4) 的，
                       # 我们通常也会把挂上去的 LoRA 层保持在 float16 精度，
                       # 这样计算更精准，生成的文本质量更高。

    is_trainable=False # 4. 开启推理模式 (Inference Mode)
                       # 非常重要！这是告诉 PEFT：“我只是想用它来生成文本，不需要再训练了。”
                       # 作用：它会确保所有参数都被设为 requires_grad=False。
                       # 好处：不构建计算图，不维护梯度，能节省大量显存，加载速度也更快。
)


c:\Users\wxb55\.conda\envs\urban_climate\llm1\lib\site-packages\peft\tuners\tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [25]:
%%time
index = 10
 
prompt = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']
formatted_prompt = f"Instruct: Summarize the following conversation.\nInput:{prompt}\nOutput:\n"
res = gen(ft_model,formatted_prompt,100,)
#print(res[0])
output = res[0].split('Output:\n')[1]
dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{formatted_prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'PEFT MODEL GENERATION:\n{output}')


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Instruct: Summarize the following conversation.
Input:#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
Output:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# attends Brian's birth